# Offline Options to Realized Variance Pipeline Demo

This notebook demonstrates one complete offline research workflow for forecasting forward realized variance from options data.

Research question:

- Can a compact set of end-of-day option-surface features improve prediction of 5-trading-day forward annualized realized variance for SPY?

Symbols used in this notebook:

- $S_t$: SPY close price at trade date $t$.
- $r_t = \ln(S_t / S_{t-1})$: daily log return.
- $RV_{t,t+5} = \sum_{i=1}^{5} r_{t+i}^2$: forward 5-day realized variance.
- $	ext{annualized RV}_{t,t+5} = (252/5) 	imes RV_{t,t+5}$: annualized forward variance.

Important runtime boundary:

- ClickHouse is **not** used at runtime in this notebook.
- The notebook reads committed files under `data/raw/` only.


In [ ]:
from pathlib import Path

import pandas as pd

from options_rv.evaluation.metrics import mean_absolute_error
from options_rv.evaluation.metrics import qlike
from options_rv.evaluation.metrics import root_mean_squared_error
from options_rv.evaluation.splits import build_chronological_split_masks
from options_rv.features.option_surface import build_option_surface_features
from options_rv.io.local_loader import load_raw_data
from options_rv.models.baselines import compute_atm_iv_baseline
from options_rv.models.baselines import compute_persistence_baseline
from options_rv.models.train import train_ridge_regression
from options_rv.pipeline.offline_research import FEATURE_COLUMNS
from options_rv.targets.realized_variance import build_forward_realized_variance_target

def _discover_project_root(start_path: Path) -> Path:
    for candidate in [start_path, *start_path.parents]:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'data' / 'raw').exists():
            return candidate
    raise RuntimeError('Could not locate project root containing pyproject.toml and data/raw')

project_root = _discover_project_root(Path.cwd().resolve())
raw_dir = project_root / 'data' / 'raw'
output_dir = project_root / 'outputs' / 'notebook_demo'

print(f'project_root={project_root}')
print(f'raw_dir_exists={raw_dir.exists()}')
print('Offline runtime only: local Parquet loader, no ClickHouse calls.')

## Raw Cache Contract

The local raw cache defines the entire runtime input boundary.

Required files:

- `options_quotes.parquet`
- `underlying_daily.parquet`
- `manifest.json`

Required options columns include symbol, dates, option type, strike, bid, ask, implied volatilities, open interest, and volume. Required underlying columns include symbol, trade date, and close.

The loader validates:

- file presence,
- schema presence,
- symbol and date coverage,
- manifest consistency.

If these checks pass, the rest of the workflow can assume deterministic inputs.


In [ ]:
raw_data = load_raw_data(raw_dir=raw_dir, required_symbol='SPY', minimum_trading_days=200)

print('options_rows=', len(raw_data.options_quotes))
print('underlying_rows=', len(raw_data.underlying_daily))
print('manifest_symbols=', raw_data.manifest['symbols'])
print('manifest_coverage=', raw_data.manifest['date_coverage'])

## Raw Data Inspection

Before feature or target engineering, inspect basic shape and coverage.

We check:

- date span,
- daily row counts,
- option type coverage,
- rough quote quality proxies (bid and ask positivity).

This keeps early sanity checks explicit and prevents hidden assumptions.


In [ ]:
options_quotes = raw_data.options_quotes.copy()
underlying_daily = raw_data.underlying_daily.copy()

summary = {
    'options_start': str(options_quotes['trade_date'].min().date()),
    'options_end': str(options_quotes['trade_date'].max().date()),
    'underlying_start': str(underlying_daily['trade_date'].min().date()),
    'underlying_end': str(underlying_daily['trade_date'].max().date()),
    'option_types': sorted(options_quotes['option_type'].unique().tolist()),
    'all_bid_positive': bool((options_quotes['bid'] > 0).all()),
    'all_ask_positive': bool((options_quotes['ask'] > 0).all()),
}

pd.Series(summary)

## Realized Variance Target Construction

Target construction follows a forward-window convention to avoid leakage.

Definitions:

- Return: $r_t = \ln(S_t/S_{t-1})$.
- Forward horizon: $h = 5$ trading days.
- Forward realized variance: $RV_{t,t+h} = \sum_{i=1}^{h} r_{t+i}^2$.
- Annualized target: $(252/h) 	imes RV_{t,t+h}$.

We also compute trailing 20-day annualized variance as a baseline feature.


In [ ]:
target_frame = build_forward_realized_variance_target(
    underlying_daily=underlying_daily,
    horizon_days=5,
    annualization_factor=252,
    use_log_target=True,
    trailing_window_days=20,
)

target_frame.head(5)

## Option Surface Feature Construction

We build a compact, explainable feature set from each symbol-date option slice.

Feature definitions:

- `atm_iv_30d`: near-30-day at-the-money implied volatility level.
- `term_slope_60d_minus_30d`: slope between near-60-day and near-30-day ATM IV.
- `downside_skew_30d`: below-spot put IV minus ATM put IV around 30 days.
- `avg_spread_ratio`: average bid-ask spread ratio proxy for quote quality.
- `total_open_interest`: liquidity proxy.
- trailing realized variance: baseline state variable from the target module.


In [ ]:
feature_frame = build_option_surface_features(
    options_quotes=options_quotes,
    underlying_daily=underlying_daily,
    trailing_variance_frame=target_frame,
)

feature_frame.head(5)

## Chronological Split Design

Forecast evaluation must respect time order.

Split policy:

- earliest observations: train,
- middle observations: validation,
- latest observations: test.

No shuffled split is used because shuffled validation can leak time information in forecasting contexts.


In [ ]:
panel = feature_frame.merge(
    target_frame[['symbol', 'trade_date', 'annualized_forward_variance_5d', 'target_log_annualized_forward_variance_5d']],
    on=['symbol', 'trade_date'],
    how='inner',
    validate='one_to_one',
)
panel = panel.dropna(subset=FEATURE_COLUMNS + ['annualized_forward_variance_5d', 'target_log_annualized_forward_variance_5d']).reset_index(drop=True)

split_labels = build_chronological_split_masks(panel)
panel['split'] = split_labels

panel['split'].value_counts()

## Baseline Forecasts

Two simple baselines set a minimum performance bar.

1. Persistence baseline: use trailing realized variance as the forecast.
2. ATM implied-vol baseline: convert ATM implied volatility to variance using $\sigma^2$.

If the main model does not improve over these, added complexity is not justified.


In [ ]:
panel['persistence_prediction'] = compute_persistence_baseline(panel)
panel['atm_iv_prediction'] = compute_atm_iv_baseline(panel)

panel[['trade_date', 'annualized_forward_variance_5d', 'persistence_prediction', 'atm_iv_prediction']].head(5)

## Main Model Training

Main model: ridge regression on standardized features.

Why ridge regression:

- linear and explainable,
- stabilizes coefficients under correlated predictors,
- strong baseline for interview-grade research narratives.

The fitted target is the log of annualized forward variance. This keeps predictions positive after exponentiation.


In [ ]:
import numpy as np
ridge_result = train_ridge_regression(
    feature_target_frame=panel,
    split_labels=split_labels,
    feature_columns=FEATURE_COLUMNS,
    target_column='target_log_annualized_forward_variance_5d',
    ridge_alpha=1.0,
)

ridge_predictions = ridge_result.predictions.copy()
panel['ridge_prediction_log'] = ridge_predictions['prediction'].to_numpy()
panel['ridge_prediction_variance'] = panel['ridge_prediction_log'].map(lambda x: float(np.exp(x)))

panel[['trade_date', 'target_log_annualized_forward_variance_5d', 'ridge_prediction_log', 'ridge_prediction_variance']].head(5)

## Evaluation

We evaluate each model in variance space using:

- Root Mean Squared Error (RMSE),
- Mean Absolute Error (MAE),
- QLIKE: $\log(f_t) + y_t/f_t$ averaged across rows, where $y_t$ is realized variance and $f_t$ is forecast variance.

Metrics are reported by split to separate in-sample behavior from out-of-sample behavior.


In [ ]:
rows = []
for split_name in ['train', 'validation', 'test']:
    split_panel = panel.loc[panel['split'] == split_name]
    y = split_panel['annualized_forward_variance_5d']
    for model_name, prediction_column in [
        ('persistence', 'persistence_prediction'),
        ('atm_iv', 'atm_iv_prediction'),
        ('ridge', 'ridge_prediction_variance'),
    ]:
        yhat = split_panel[prediction_column]
        rows.append({
            'split': split_name,
            'model': model_name,
            'rmse': root_mean_squared_error(y, yhat),
            'mae': mean_absolute_error(y, yhat),
            'qlike': qlike(y, yhat),
        })

metrics = pd.DataFrame(rows)
metrics

## Coefficient and Feature Interpretation

Ridge coefficients are estimated in standardized feature space.

Interpretation rule:

- positive coefficient: higher feature value is associated with higher log forward variance,
- negative coefficient: higher feature value is associated with lower log forward variance,
- larger absolute value: stronger association under this linear specification.

This is association, not a structural causal claim.


In [ ]:
coefficients = ridge_result.coefficients.sort_values('coefficient', key=lambda s: s.abs(), ascending=False).reset_index(drop=True)
coefficients

## Limitations and Next Steps

Current scope is intentionally narrow.

Limitations:

- one symbol (SPY),
- one horizon (5 days),
- one linear model,
- compact feature set,
- synthetic or heavily filtered offline cache may differ from full production-quality market data.

Natural next steps:

- extend robustness checks across adjacent horizons,
- stress-test feature stability over subsamples,
- compare with another simple linear alternative while preserving explainability.


In [ ]:
output_dir.mkdir(parents=True, exist_ok=True)
metrics.to_csv(output_dir / 'notebook_metrics.csv', index=False)
coefficients.to_csv(output_dir / 'notebook_coefficients.csv', index=False)

print('Saved notebook artifacts:')
print('-', output_dir / 'notebook_metrics.csv')
print('-', output_dir / 'notebook_coefficients.csv')